# Mask Inventory Problem in Pure Python
## Exact Techniques for Book Problem 3.3

This notebook solves the mask-production planning problem from book section `3.3` in pure Python.

Because normal production has no explicit cost in the model, the optimization depends only on:
- extra production cost,
- end-of-month inventory cost,
- the inventory-balance equations.

We compare three exact approaches:
1. explicit search over feasible inventory trajectories,
2. memoized dynamic programming,
3. verification of the published monthly plan.


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from time import perf_counter


In [2]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


MONTHS = [1, 2, 3, 4]
DEMAND = {1: 2800, 2: 2200, 3: 3200, 4: 2500}
NORMAL_CAPACITY = 2700
EXTRA_CAPACITY = 300
EXTRA_COST = 10
INVENTORY_COST = 2
INVENTORY_BOUNDS = {1: 200, 2: 800, 3: 300, 4: 0}
EXPECTED = {
    "normal": {1: 2700, 2: 2700, 3: 2700, 4: 2500},
    "extra": {1: 100, 2: 0, 3: 0, 4: 0},
    "inventory": {0: 0, 1: 0, 2: 500, 3: 0, 4: 0},
    "cost": 2_000,
}


def plan_from_inventory(inventory):
    normal = {}
    extra = {}
    total_cost = 0
    for month in MONTHS:
        required_production = DEMAND[month] + inventory[month] - inventory[month - 1]
        if required_production < 0 or required_production > NORMAL_CAPACITY + EXTRA_CAPACITY:
            return None
        extra_units = max(0, required_production - NORMAL_CAPACITY)
        normal_units = required_production - extra_units
        if normal_units > NORMAL_CAPACITY or extra_units > EXTRA_CAPACITY:
            return None
        normal[month] = normal_units
        extra[month] = extra_units
        total_cost += EXTRA_COST * extra_units + INVENTORY_COST * inventory[month]
    return {"normal": normal, "extra": extra, "inventory": inventory, "cost": total_cost}


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["cost"], tuple(left["inventory"][month] for month in MONTHS))
    right_key = (right["cost"], tuple(right["inventory"][month] for month in MONTHS))
    return right if right_key < left_key else left


## 1. Explicit Search on Inventory Trajectories


In [3]:
@timed
def solve_mask_inventory_search():
    best = None
    for inventory_1 in range(INVENTORY_BOUNDS[1] + 1):
        for inventory_2 in range(INVENTORY_BOUNDS[2] + 1):
            inventory = {0: 0, 1: inventory_1, 2: inventory_2, 3: 0, 4: 0}
            best = choose_better(best, plan_from_inventory(inventory))
    return best


search_result, search_time = solve_mask_inventory_search()
print("Inventory-search result:", search_result)
print(f"Elapsed time: {search_time:.6f} seconds")
assert search_result == EXPECTED


Inventory-search result: {'normal': {1: 2700, 2: 2700, 3: 2700, 4: 2500}, 'extra': {1: 100, 2: 0, 3: 0, 4: 0}, 'inventory': {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}, 'cost': 2000}
Elapsed time: 0.274518 seconds


## 2. Dynamic Programming


In [4]:
@timed
def solve_mask_inventory_dynamic_programming():
    @lru_cache(maxsize=None)
    def dp(month, previous_inventory):
        if month == 5:
            return (0, ()) if previous_inventory == 0 else None

        best = None
        for inventory_t in range(INVENTORY_BOUNDS[month] + 1):
            required_production = DEMAND[month] + inventory_t - previous_inventory
            if required_production < 0 or required_production > NORMAL_CAPACITY + EXTRA_CAPACITY:
                continue
            extra_units = max(0, required_production - NORMAL_CAPACITY)
            normal_units = required_production - extra_units
            if normal_units > NORMAL_CAPACITY or extra_units > EXTRA_CAPACITY:
                continue
            tail = dp(month + 1, inventory_t)
            if tail is None:
                continue
            total_cost = EXTRA_COST * extra_units + INVENTORY_COST * inventory_t + tail[0]
            candidate = (total_cost, ((normal_units, extra_units, inventory_t),) + tail[1])
            if best is None or candidate[0] < best[0] or (candidate[0] == best[0] and candidate[1] < best[1]):
                best = candidate
        return best

    packed = dp(1, 0)
    inventory = {0: 0}
    normal = {}
    extra = {}
    for month, (normal_units, extra_units, inventory_t) in enumerate(packed[1], start=1):
        normal[month] = normal_units
        extra[month] = extra_units
        inventory[month] = inventory_t
    return {"normal": normal, "extra": extra, "inventory": inventory, "cost": packed[0]}


dp_result, dp_time = solve_mask_inventory_dynamic_programming()
print("Dynamic-programming result:", dp_result)
print(f"Elapsed time: {dp_time:.6f} seconds")
assert dp_result == EXPECTED


Dynamic-programming result: {'normal': {1: 2700, 2: 2700, 3: 2700, 4: 2500}, 'extra': {1: 100, 2: 0, 3: 0, 4: 0}, 'inventory': {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}, 'cost': 2000}
Elapsed time: 0.115047 seconds


## 3. Published Textbook Plan


In [5]:
textbook_inventory = {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}
textbook_result = plan_from_inventory(textbook_inventory)
print("Textbook result:", textbook_result)

assert textbook_result == EXPECTED
assert search_result == dp_result == textbook_result
print("\nAll Python techniques agree with the published production plan.")


Textbook result: {'normal': {1: 2700, 2: 2700, 3: 2700, 4: 2500}, 'extra': {1: 100, 2: 0, 3: 0, 4: 0}, 'inventory': {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}, 'cost': 2000}

All Python techniques agree with the published production plan.
